# Лабораторная 5: полный трансформер `encoder-decoder` на `Tiny Shakespeare` (starter)

Цель: реализовать полный контур `encoder-decoder` для предсказания следующего токена
и пройти детерминированный workflow `TODO 1..6`.


## Маршрут выполнения

1. `TODO 1`: контракт данных.
2. `TODO 2`: ручной пример причинной маски.
3. `TODO 3`: модель `encoder-decoder`.
4. `TODO 4`: обучение, `loss`, `perplexity`.
5. `TODO 5`: детерминированная генерация.
6. `TODO 6`: диагностика внимания.


In [1]:
import ctypes
import json
import os
import random
import subprocess
import sys
import time
from pathlib import Path

PROFILES = {
    'CPU-friendly': {
        'max_chars': 140_000,
        'src_len': 72,
        'tgt_len': 72,
        'stride': 3,
        'batch_size': 64,
        'd_model': 128,
        'num_heads': 4,
        'd_ff': 256,
        'num_layers': 2,
        'dropout': 0.1,
        'learning_rate': 3e-4,
        'epochs': 14,
        'early_stopping_patience': 3,
        'probe_count': 20,
        'gen_len': 48,
        'gen_success_threshold': 18,
        'gen_mean_threshold': 0.70,
    },
    'GPU-friendly': {
        'max_chars': 220_000,
        'src_len': 80,
        'tgt_len': 80,
        'stride': 2,
        'batch_size': 128,
        'd_model': 192,
        'num_heads': 6,
        'd_ff': 512,
        'num_layers': 3,
        'dropout': 0.1,
        'learning_rate': 2.5e-4,
        'epochs': 18,
        'early_stopping_patience': 3,
        'probe_count': 20,
        'gen_len': 56,
        'gen_success_threshold': 18,
        'gen_mean_threshold': 0.70,
    },
}

RUNTIME_PROFILE = os.getenv('COURSE_RUNTIME_PROFILE', 'CPU-friendly')
if RUNTIME_PROFILE not in PROFILES:
    raise ValueError(f'Неизвестный профиль: {RUNTIME_PROFILE}')

SEED = 42
RUN_STARTED_AT = time.time()


def _prepend_path_env(var_name: str, new_paths: list[Path]) -> None:
    """Добавляет пути в начало переменной окружения.

    Аргументы:
        var_name: Имя переменной окружения с путями.
        new_paths: Пути-кандидаты для добавления.

    Возвращает:
        None.

    Исключения:
        RuntimeError: Не выбрасывается в штатном режиме.
    """
    existing = os.environ.get(var_name, '')
    existing_parts = [part for part in existing.split(':') if part]
    merged: list[str] = []
    for path in new_paths:
        if path.is_dir():
            path_str = str(path)
            if path_str not in merged:
                merged.append(path_str)
    for part in existing_parts:
        if part not in merged:
            merged.append(part)
    if merged:
        os.environ[var_name] = ':'.join(merged)


def _detect_site_packages_dir() -> Path | None:
    """Возвращает каталог `site-packages` активной виртуальной среды.

    Аргументы:
        Нет.

    Возвращает:
        Путь к `site-packages` или `None`.

    Исключения:
        RuntimeError: Не выбрасывается в штатном режиме.
    """
    major, minor = sys.version_info[:2]
    candidate = Path(sys.prefix) / 'lib' / f'python{major}.{minor}' / 'site-packages'
    if candidate.is_dir():
        return candidate
    return None


def _preload_cuda_runtime_libraries(site_packages: Path) -> dict:
    """Предзагружает CUDA-библиотеки до импорта TensorFlow.

    Аргументы:
        site_packages: Каталог `site-packages` текущей среды.

    Возвращает:
        Словарь с ключами `loaded`, `missing`, `failed`.

    Исключения:
        RuntimeError: Не выбрасывается в штатном режиме.
    """
    nvidia_root = site_packages / 'nvidia'
    library_specs = [
        ('cuda_runtime', 'libcudart.so.12'),
        ('cublas', 'libcublas.so.12'),
        ('cublas', 'libcublasLt.so.12'),
        ('cudnn', 'libcudnn.so.9'),
        ('cufft', 'libcufft.so.11'),
        ('curand', 'libcurand.so.10'),
        ('cusolver', 'libcusolver.so.11'),
        ('cusparse', 'libcusparse.so.12'),
        ('nccl', 'libnccl.so.2'),
        ('nvjitlink', 'libnvJitLink.so.12'),
    ]

    loaded: list[str] = []
    missing: list[str] = []
    failed: list[str] = []

    for subdir, library_name in library_specs:
        library_path = nvidia_root / subdir / 'lib' / library_name
        if not library_path.is_file():
            missing.append(str(library_path))
            continue
        try:
            ctypes.CDLL(str(library_path), mode=ctypes.RTLD_GLOBAL)
            loaded.append(str(library_path))
        except OSError as exc:
            failed.append(f'{library_path}: {exc}')

    return {'loaded': loaded, 'missing': missing, 'failed': failed}


def _configure_local_gpu_runtime_env(runtime_profile: str) -> dict:
    """Готовит переменные окружения для локального запуска на GPU.

    Аргументы:
        runtime_profile: Текущий профиль (`CPU-friendly` или `GPU-friendly`).

    Возвращает:
        Сводка применения настройки окружения.

    Исключения:
        RuntimeError: Не выбрасывается в штатном режиме.
    """
    if runtime_profile != 'GPU-friendly':
        return {'applied': False, 'reason': 'runtime_profile != GPU-friendly'}

    site_packages = _detect_site_packages_dir()
    if site_packages is None:
        return {'applied': False, 'reason': 'site-packages not found'}

    tensorflow_dir = site_packages / 'tensorflow'
    nvidia_root = site_packages / 'nvidia'
    nvidia_lib_dirs = sorted(path for path in nvidia_root.glob('*/lib') if path.is_dir())

    _prepend_path_env('LD_LIBRARY_PATH', [tensorflow_dir, *nvidia_lib_dirs])
    _prepend_path_env('PATH', [site_packages / 'nvidia' / 'cuda_nvcc' / 'bin'])

    preload_report = _preload_cuda_runtime_libraries(site_packages)

    return {
        'applied': True,
        'site_packages': str(site_packages),
        'tensorflow_dir': str(tensorflow_dir),
        'nvidia_lib_dirs': [str(path) for path in nvidia_lib_dirs],
        'preload_report': preload_report,
    }


gpu_env_info = _configure_local_gpu_runtime_env(RUNTIME_PROFILE)

# import matplotlib.pyplot as plt  # временно отключено из-за ошибки установки
import numpy as np
import tensorflow as tf
from tensorflow import keras


def seed_everything(seed: int) -> None:
    """Фиксирует источники случайности.

    Аргументы:
        seed: Целое зерно случайности.

    Возвращает:
        None.

    Исключения:
        ValueError: Если `seed` отрицательный.
    """
    if seed < 0:
        raise ValueError('seed должен быть неотрицательным.')
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def gpu_preflight(runtime_profile: str) -> dict:
    """Проверяет реальную готовность GPU-пути перед обучением.

    Аргументы:
        runtime_profile: Выбранный профиль выполнения.

    Возвращает:
        Краткий отчёт о найденных GPU и версии TensorFlow.

    Исключения:
        RuntimeError: Если для `GPU-friendly` не выполнены условия запуска на GPU.
    """
    if runtime_profile != 'GPU-friendly':
        return {
            'passed': False,
            'skipped': True,
            'reason': 'runtime_profile != GPU-friendly',
        }

    try:
        nvidia_report = subprocess.run(
            ['nvidia-smi', '-L'],
            check=True,
            capture_output=True,
            text=True,
        )
        lines = [line for line in nvidia_report.stdout.strip().splitlines() if line.strip()]
        print('nvidia-smi -L (первые строки):')
        for line in lines[:3]:
            print('  ', line)
    except Exception as exc:
        raise RuntimeError(
            'GPU не виден (setup): команда nvidia-smi недоступна или вернула ошибку. '
            'Используйте готовый маршрут из '
            'themes/00-Foundations/guides/05_local_tensorflow_gpu_notebooks.md.'
        ) from exc

    print('TensorFlow version:', tf.__version__)
    print('TensorFlow built with CUDA:', tf.test.is_built_with_cuda())
    build_info = tf.sysconfig.get_build_info()
    print('TensorFlow build cuda_version:', build_info.get('cuda_version', 'unknown'))
    print('TensorFlow build cudnn_version:', build_info.get('cudnn_version', 'unknown'))

    physical_gpus = tf.config.list_physical_devices('GPU')
    logical_gpus = tf.config.list_logical_devices('GPU')
    print('Physical GPUs:', [device.name for device in physical_gpus])
    print('Logical GPUs :', [device.name for device in logical_gpus])

    if len(physical_gpus) == 0 or len(logical_gpus) == 0:
        raise RuntimeError(
            'GPU не виден (setup): TensorFlow не зарегистрировал GPU-устройства. '
            'Проверьте окружение по guides 05/06/07 в themes/00-Foundations.'
        )

    try:
        with tf.device('/GPU:0'):
            lhs = tf.random.normal((128, 128), dtype=tf.float32)
            rhs = tf.random.normal((128, 128), dtype=tf.float32)
            product = tf.matmul(lhs, rhs)
            _ = float(tf.reduce_mean(product).numpy())

        with tf.device('/GPU:0'):
            smoke_model = keras.Sequential([
                keras.layers.Input(shape=(16,), dtype=tf.float32),
                keras.layers.Dense(32, activation='relu'),
                keras.layers.Dense(1),
            ])
            smoke_model.compile(optimizer='adam', loss='mse')
            features = tf.random.normal((32, 16), dtype=tf.float32)
            targets = tf.random.normal((32, 1), dtype=tf.float32)
            smoke_model.train_on_batch(features, targets)
    except Exception as exc:
        raise RuntimeError(
            'GPU виден, но kernels падают (compatibility). Это не ошибка кода ЛР. '
            'См. themes/00-Foundations/guides/07_tensorflow_blackwell_local_gpu_case_study.md. '
            f'Исходная ошибка: {type(exc).__name__}: {exc}'
        ) from exc

    print('gpu_preflight(): PASSED')
    return {
        'passed': True,
        'physical_gpus': [device.name for device in physical_gpus],
        'logical_gpus': [device.name for device in logical_gpus],
        'tf_version': tf.__version__,
    }


cfg = dict(PROFILES[RUNTIME_PROFILE])
cfg['runtime_profile'] = RUNTIME_PROFILE
seed_everything(SEED)
preflight_info = gpu_preflight(RUNTIME_PROFILE)

print(json.dumps(cfg, ensure_ascii=False, indent=2))
print('gpu_env_info:', json.dumps(gpu_env_info, ensure_ascii=False, indent=2)[:800])
print('preflight_info:', json.dumps(preflight_info, ensure_ascii=False, indent=2))




{
  "max_chars": 140000,
  "src_len": 72,
  "tgt_len": 72,
  "stride": 3,
  "batch_size": 64,
  "d_model": 128,
  "num_heads": 4,
  "d_ff": 256,
  "num_layers": 2,
  "dropout": 0.1,
  "learning_rate": 0.0003,
  "epochs": 14,
  "early_stopping_patience": 3,
  "probe_count": 20,
  "gen_len": 48,
  "gen_success_threshold": 18,
  "gen_mean_threshold": 0.7,
  "runtime_profile": "CPU-friendly"
}
gpu_env_info: {
  "applied": false,
  "reason": "runtime_profile != GPU-friendly"
}
preflight_info: {
  "passed": false,
  "skipped": true,
  "reason": "runtime_profile != GPU-friendly"
}


## TODO 1. Контракт данных

Реализуйте загрузку `Tiny Shakespeare`, токенизацию и окна:
- `encoder_input = ids[i : i + SRC_LEN]`
- `target = ids[i + SRC_LEN : i + SRC_LEN + TGT_LEN]`
- `decoder_input = [SOS] + target[:-1]`
- `decoder_target = target`

**Смысл блока:** превратить непрерывный текст в воспроизводимые учебные пары «контекст -> продолжение».

**Что обязательно проверить:** формы тензоров, корректный сдвиг через `SOS`, совпадение первых `k` примеров при одном и том же `seed`.


In [2]:
# ============================================================
# TODO 1. Контракт данных
# ============================================================
PAD_TOKEN = '<pad>'
SOS_TOKEN = '<sos>'


# --- TODO 1.1 ---
def load_tiny_shakespeare_text(profile_cfg: dict) -> str:
    """Загружает корпус Tiny Shakespeare.

    Аргументы:
        profile_cfg: Конфигурация активного профиля.

    Возвращает:
        Текст корпуса.

    Исключения:
        RuntimeError: Если загрузка корпуса завершилась ошибкой.
    """
    path_to_file = tf.keras.utils.get_file(
        'shakespeare.txt',
        origin='https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        cache_dir='.',
        cache_subdir='data',
    )
    text = Path(path_to_file).read_text()
    text = text[: profile_cfg['max_chars']]
    return text


# --- TODO 1.2 ---
def build_char_vocabulary(text: str) -> tuple[list[str], dict[str, int], dict[int, str]]:
    """Строит символьный словарь.

    Аргументы:
        text: Исходный текст корпуса.

    Возвращает:
        `(vocab, char_to_id, id_to_char)`.

    Исключения:
        ValueError: Если текст пустой.
    """
    if len(text) == 0:
        raise ValueError('Текст пустой, не могу построить словарь.')
    chars = sorted(set(text))
    vocab = [PAD_TOKEN, SOS_TOKEN] + chars
    char_to_id = {ch: idx for idx, ch in enumerate(vocab)}
    id_to_char = {idx: ch for idx, ch in enumerate(vocab)}
    return vocab, char_to_id, id_to_char


# --- TODO 1.3 ---
def encode_text(text: str, char_to_id: dict[str, int]) -> np.ndarray:
    """Кодирует текст в индексы.

    Аргументы:
        text: Исходный текст.
        char_to_id: Словарь символ -> индекс.

    Возвращает:
        Массив индексов.

    Исключения:
        KeyError: Если символ отсутствует в словаре.
    """
    return np.array([char_to_id[ch] for ch in text], dtype=np.int32)


# --- TODO 1.4 ---
def build_seq2seq_windows(
    token_ids: np.ndarray,
    src_len: int,
    tgt_len: int,
    stride: int,
    sos_id: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Формирует окна для `encoder/decoder`.

    Аргументы:
        token_ids: Полная токенизированная последовательность.
        src_len: Длина входа кодировщика.
        tgt_len: Длина цели декодера.
        stride: Шаг окна.
        sos_id: Индекс `SOS`.

    Возвращает:
        `(encoder_inputs, decoder_inputs, decoder_targets)`.

    Исключения:
        ValueError: Если корпус слишком короткий.
    """
    window_size = src_len + tgt_len
    if len(token_ids) < window_size:
        raise ValueError(
            f'Корпус слишком короткий: длина {len(token_ids)}, '
            f'требуется минимум {window_size} токенов.'
        )

    encoder_inputs_list = []
    decoder_inputs_list = []
    decoder_targets_list = []

    for i in range(0, len(token_ids) - window_size + 1, stride):
        enc_inp = token_ids[i : i + src_len]
        target = token_ids[i + src_len : i + src_len + tgt_len]
        dec_inp = np.empty(tgt_len, dtype=np.int32)
        dec_inp[0] = sos_id
        dec_inp[1:] = target[:-1]
        dec_tgt = target

        encoder_inputs_list.append(enc_inp)
        decoder_inputs_list.append(dec_inp)
        decoder_targets_list.append(dec_tgt)

    encoder_inputs = np.stack(encoder_inputs_list, axis=0)
    decoder_inputs = np.stack(decoder_inputs_list, axis=0)
    decoder_targets = np.stack(decoder_targets_list, axis=0)

    return encoder_inputs, decoder_inputs, decoder_targets


# --- TODO 1.5: Загрузка, токенизация, окна, датасеты ---
full_text = load_tiny_shakespeare_text(cfg)
vocab, char_to_id, id_to_char = build_char_vocabulary(full_text)
vocab_size = len(vocab)
pad_id = char_to_id[PAD_TOKEN]
sos_id = char_to_id[SOS_TOKEN]

all_ids = encode_text(full_text, char_to_id)
print(f'Длина текста: {len(full_text)} символов')
print(f'Размер словаря: {vocab_size}')
print(f'Количество токенов: {len(all_ids)}')

# Разбиение: 80% train, 10% val, 10% test
n_total = len(all_ids)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)

train_ids = all_ids[:n_train]
val_ids = all_ids[n_train : n_train + n_val]
test_ids = all_ids[n_train + n_val :]
print(f'Train токенов: {len(train_ids)}, Val токенов: {len(val_ids)}, Test токенов: {len(test_ids)}')

# Окна
train_enc, train_dec_inp, train_dec_tgt = build_seq2seq_windows(
    train_ids, cfg['src_len'], cfg['tgt_len'], cfg['stride'], sos_id
)
val_enc, val_dec_inp, val_dec_tgt = build_seq2seq_windows(
    val_ids, cfg['src_len'], cfg['tgt_len'], cfg['stride'], sos_id
)
test_enc, test_dec_inp, test_dec_tgt = build_seq2seq_windows(
    test_ids, cfg['src_len'], cfg['tgt_len'], cfg['stride'], sos_id
)
print(f'Train окон: {len(train_enc)}, Val окон: {len(val_enc)}, Test окон: {len(test_enc)}')

# Имена для Mini-check 1
train_encoder_inputs = train_enc
train_decoder_inputs = train_dec_inp
train_decoder_targets = train_dec_tgt

# tf.data.Dataset
batch_size = cfg['batch_size']

train_dataset = (
    tf.data.Dataset.from_tensor_slices(
        (
            {'encoder_inputs': train_enc, 'decoder_inputs': train_dec_inp},
            train_dec_tgt,
        )
    )
    .shuffle(10_000, seed=SEED, reshuffle_each_iteration=True)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.Dataset.from_tensor_slices(
        (
            {'encoder_inputs': val_enc, 'decoder_inputs': val_dec_inp},
            val_dec_tgt,
        )
    )
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    tf.data.Dataset.from_tensor_slices(
        (
            {'encoder_inputs': test_enc, 'decoder_inputs': test_dec_inp},
            test_dec_tgt,
        )
    )
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


# --- TODO 1.6: probe_pairs ---
probe_indices = list(
    range(0, len(test_enc), max(1, len(test_enc) // cfg['probe_count']))
)[: cfg['probe_count']]

probe_pairs = [
    {
        'encoder_input': test_enc[idx],
        'decoder_target': test_dec_tgt[idx],
    }
    for idx in probe_indices
]
print(f'Зафиксировано probe_pairs: {len(probe_pairs)}')




1115394/1115394 [==============================] - 1s 1us/step
Длина текста: 140000 символов
Размер словаря: 63
Количество токенов: 140000
Train токенов: 112000, Val токенов: 14000, Test токенов: 14000
Train окон: 37286, Val окон: 4619, Test окон: 4619
Зафиксировано probe_pairs: 20


In [3]:
# Мини-проверка 1
# Смысл: быстрый контроль, что контракт данных реализован до перехода к модели.
# Частая ошибка: перепутаны оси `(batch, length)` и `(length, batch)`.
assert 'train_encoder_inputs' in globals(), 'Ожидается train_encoder_inputs.'
assert 'train_decoder_inputs' in globals(), 'Ожидается train_decoder_inputs.'
assert 'train_decoder_targets' in globals(), 'Ожидается train_decoder_targets.'

assert train_encoder_inputs.shape[1] == cfg['src_len']
assert train_decoder_inputs.shape[1] == cfg['tgt_len']
assert train_decoder_targets.shape[1] == cfg['tgt_len']

assert len(probe_pairs) == cfg['probe_count']
print('Mini-check 1 пройден.')


Mini-check 1 пройден.


## TODO 2. Ручной пример причинной маски

**Смысл блока:** увидеть, что декодер не имеет права смотреть вправо по времени (в будущее).

**Что обязательно проверить:** маска нижнетреугольная, диагональ разрешена, выше диагонали только запрет.


In [4]:
# ============================================================
# TODO 2. Ручной пример причинной маски
# ============================================================

def build_causal_mask(seq_len: int) -> np.ndarray:
    """Строит нижнетреугольную причинную маску.

    Аргументы:
        seq_len: Длина последовательности.

    Возвращает:
        Булева матрица `(seq_len, seq_len)`.

    Исключения:
        ValueError: Если `seq_len` неположителен.
    """
    if seq_len <= 0:
        raise ValueError(f'seq_len должен быть положительным, получено {seq_len}')

    # Нижнетреугольная матрица: True на диагонали и ниже, False выше
    mask = np.tril(np.ones((seq_len, seq_len), dtype=bool))
    return mask


# --- TODO 2.2: Проверка ---
# Проверяем для seq_len = 5
demo_len = 5
causal_mask = build_causal_mask(demo_len)

# 1. Форма
assert causal_mask.shape == (demo_len, demo_len), (
    f'Ожидалась форма ({demo_len}, {demo_len}), получена {causal_mask.shape}'
)

# 2. Нижнетреугольность: диагональ и ниже — True
for i in range(demo_len):
    for j in range(demo_len):
        if j <= i:
            assert causal_mask[i, j] == True, (
                f'Позиция [{i},{j}] должна быть True (диагональ/ниже)'
            )
        else:
            assert causal_mask[i, j] == False, (
                f'Позиция [{i},{j}] должна быть False (выше диагонали — будущее)'
            )

print('Форма маски:', causal_mask.shape)
print('Маска (5×5):\n', causal_mask.astype(int))
print('Mini-check 2 пройден: маска нижнетреугольная, будущее запрещено.')

Форма маски: (5, 5)
Маска (5×5):
 [[1 0 0 0 0]
 [1 1 0 0 0]
 [1 1 1 0 0]
 [1 1 1 1 0]
 [1 1 1 1 1]]
Mini-check 2 пройден: маска нижнетреугольная, будущее запрещено.


## TODO 3. Реализация модели `encoder-decoder`

**Смысл блока:** собрать полный тракт `кодировщик -> декодер` и связать маски с реальными тензорами внимания.

**Что обязательно проверить:** формы выходов каждого блока, корректность `encoder/decoder/cross` масок, один успешный `forward pass`.


In [10]:
# ============================================================
# TODO 3. Реализация модели encoder-decoder (исправленная)
# ============================================================

# --- TODO 3.1: Синусоидальное позиционное кодирование ---
def sinusoidal_position_encoding(length: int, depth: int) -> tf.Tensor:
    """Возвращает синусоидальное позиционное кодирование."""
    if length <= 0 or depth <= 0:
        raise ValueError(f'length и depth должны быть положительными, получено length={length}, depth={depth}')

    positions = np.arange(length)[:, np.newaxis]
    i = np.arange(depth)[np.newaxis, :]

    angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(depth))
    angle_rads = positions * angle_rates

    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    return tf.cast(angle_rads, dtype=tf.float32)


# --- TODO 3.2 + 3.3: TokenAndPositionEmbedding ---
class TokenAndPositionEmbedding(keras.layers.Layer):
    """Токенные эмбеддинги + позиционное кодирование."""

    def __init__(self, vocab_size: int, d_model: int, max_len: int) -> None:
        super().__init__()
        if vocab_size <= 0 or d_model <= 0 or max_len <= 0:
            raise ValueError('vocab_size, d_model и max_len должны быть положительными.')
        self.token_embedding = keras.layers.Embedding(
            input_dim=vocab_size, output_dim=d_model, mask_zero=False
        )
        self.position_encoding = sinusoidal_position_encoding(max_len, d_model)
        self.d_model = d_model

    def call(self, token_ids: tf.Tensor) -> tf.Tensor:
        token_emb = self.token_embedding(token_ids)
        length = tf.shape(token_ids)[1]
        pos_enc = self.position_encoding[:length, :]
        return token_emb + pos_enc


# --- TODO 3.4 + 3.5: TransformerEncoderBlock ---
class TransformerEncoderBlock(keras.layers.Layer):
    """Один блок кодировщика трансформера."""

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout_rate: float) -> None:
        super().__init__()
        self.self_attention = keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout_rate
        )
        self.ffn = keras.Sequential([
            keras.layers.Dense(d_ff, activation='relu'),
            keras.layers.Dense(d_model),
        ])
        self.layer_norm1 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.layer_norm2 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = keras.layers.Dropout(dropout_rate)
        self.dropout2 = keras.layers.Dropout(dropout_rate)

    def call(self, x: tf.Tensor, attention_mask: tf.Tensor, training: bool) -> tf.Tensor:
        attn_output = self.self_attention(
            query=x, key=x, value=x, attention_mask=attention_mask, training=training
        )
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layer_norm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layer_norm2(out1 + ffn_output)

        return out2


# --- TODO 3.6 + 3.7: TransformerDecoderBlock ---
class TransformerDecoderBlock(keras.layers.Layer):
    """Один блок декодера: masked self-attention + cross-attention."""

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout_rate: float) -> None:
        super().__init__()
        self.self_attention = keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout_rate
        )
        self.cross_attention = keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout_rate
        )
        self.ffn = keras.Sequential([
            keras.layers.Dense(d_ff, activation='relu'),
            keras.layers.Dense(d_model),
        ])
        self.layer_norm1 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.layer_norm2 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.layer_norm3 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = keras.layers.Dropout(dropout_rate)
        self.dropout2 = keras.layers.Dropout(dropout_rate)
        self.dropout3 = keras.layers.Dropout(dropout_rate)

    def call(
        self,
        x: tf.Tensor,
        encoder_output: tf.Tensor,
        self_attention_mask: tf.Tensor,
        cross_attention_mask: tf.Tensor,
        training: bool,
        return_attention_scores: bool = False,
    ) -> tuple[tf.Tensor, tf.Tensor | None]:
        self_attn_output = self.self_attention(
            query=x, key=x, value=x, attention_mask=self_attention_mask, training=training
        )
        self_attn_output = self.dropout1(self_attn_output, training=training)
        out1 = self.layer_norm1(x + self_attn_output)

        cross_attn_output = self.cross_attention(
            query=out1,
            key=encoder_output,
            value=encoder_output,
            attention_mask=cross_attention_mask,
            training=training,
        )
        cross_attn_output = self.dropout2(cross_attn_output, training=training)
        out2 = self.layer_norm2(out1 + cross_attn_output)

        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        out3 = self.layer_norm3(out2 + ffn_output)

        attention_scores = None
        if return_attention_scores:
            _, attention_scores = self.self_attention(
                query=x, key=x, value=x,
                attention_mask=self_attention_mask,
                training=training,
                return_attention_scores=True,
            )

        return out3, attention_scores


# --- TODO 3.8 + 3.9: FullTransformerModel ---
class FullTransformerModel(keras.Model):
    """Полный encoder-decoder трансформер."""

    def __init__(
        self,
        vocab_size: int,
        src_len: int,
        tgt_len: int,
        d_model: int,
        num_heads: int,
        d_ff: int,
        num_layers: int,
        dropout_rate: float,
        pad_id: int,
    ) -> None:
        super().__init__()
        self.src_len = src_len
        self.tgt_len = tgt_len
        self.d_model = d_model
        self.pad_id = pad_id

        self.encoder_embedding = TokenAndPositionEmbedding(vocab_size, d_model, max_len=src_len)
        self.decoder_embedding = TokenAndPositionEmbedding(vocab_size, d_model, max_len=tgt_len)

        self.encoder_blocks = [
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout_rate)
            for _ in range(num_layers)
        ]
        self.decoder_blocks = [
            TransformerDecoderBlock(d_model, num_heads, d_ff, dropout_rate)
            for _ in range(num_layers)
        ]

        self.dropout = keras.layers.Dropout(dropout_rate)
        self.final_dense = keras.layers.Dense(vocab_size)

    def _build_padding_mask(self, token_ids: tf.Tensor) -> tf.Tensor:
        """Padding mask: True для PAD (запрет внимания)."""
        return tf.cast(tf.equal(token_ids, self.pad_id), tf.float32)[:, tf.newaxis, tf.newaxis, :]

    def _build_causal_mask(self, seq_len: tf.Tensor) -> tf.Tensor:
        """Причинная маска: запрет будущего (True = запрещено)."""
        return 1.0 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)

    def call(
        self,
        inputs,
        training: bool = False,
        return_attention: bool = False,
    ):
        """Принимает словарь {'encoder_inputs': ..., 'decoder_inputs': ...} или кортеж."""
        # Поддержка словаря и кортежа
        if isinstance(inputs, dict):
            encoder_tokens = inputs['encoder_inputs']
            decoder_tokens = inputs['decoder_inputs']
        else:
            encoder_tokens, decoder_tokens = inputs

        # --- Маски ---
        enc_padding_mask = self._build_padding_mask(encoder_tokens)
        dec_padding_mask = self._build_padding_mask(decoder_tokens)
        causal_mask = self._build_causal_mask(tf.shape(decoder_tokens)[1])
        dec_self_mask = tf.maximum(dec_padding_mask, causal_mask)
        cross_mask = enc_padding_mask

        # --- Encoder ---
        enc_x = self.encoder_embedding(encoder_tokens)
        enc_x = self.dropout(enc_x, training=training)

        for enc_block in self.encoder_blocks:
            enc_x = enc_block(enc_x, enc_padding_mask, training=training)

        encoder_output = enc_x

        # --- Decoder ---
        dec_x = self.decoder_embedding(decoder_tokens)
        dec_x = self.dropout(dec_x, training=training)

        attention_scores = None
        for i, dec_block in enumerate(self.decoder_blocks):
            is_last = i == len(self.decoder_blocks) - 1
            dec_x, attn = dec_block(
                dec_x,
                encoder_output,
                dec_self_mask,
                cross_mask,
                training=training,
                return_attention_scores=(return_attention and is_last),
            )
            if attn is not None:
                attention_scores = attn

        # --- Проекция ---
        logits = self.final_dense(dec_x)

        if return_attention:
            return logits, attention_scores
        return logits


# --- TODO 3.10: Mini-check 3 ---
model = FullTransformerModel(
    vocab_size=vocab_size,
    src_len=cfg['src_len'],
    tgt_len=cfg['tgt_len'],
    d_model=cfg['d_model'],
    num_heads=cfg['num_heads'],
    d_ff=cfg['d_ff'],
    num_layers=cfg['num_layers'],
    dropout_rate=cfg['dropout'],
    pad_id=pad_id,
)

for batch_inputs, batch_targets in train_dataset.take(1):
    batch_logits = model(batch_inputs, training=False)
    print('Форма входных токенов энкодера:', batch_inputs['encoder_inputs'].shape)
    print('Форма входных токенов декодера:', batch_inputs['decoder_inputs'].shape)
    print('Форма логитов:', batch_logits.shape)
    assert batch_logits.shape[0] == batch_inputs['encoder_inputs'].shape[0]
    assert batch_logits.shape[1] == cfg['tgt_len']
    assert batch_logits.shape[2] == vocab_size
    print('Mini-check 3 пройден: forward pass успешен, форма логитов корректна.')
    break

Форма входных токенов энкодера: (64, 72)
Форма входных токенов декодера: (64, 72)
Форма логитов: (64, 72, 63)
Mini-check 3 пройден: forward pass успешен, форма логитов корректна.


## TODO 4. Обучение и перплексия

**Смысл блока:** связать математику `NLL -> cross-entropy -> perplexity` с фактическими вычислениями в коде.

**Что обязательно проверить:** маскирование `PAD`, динамика `train/val loss`, неравенство `test_perplexity < baseline_perplexity`.


In [12]:
# ============================================================
# TODO 4. Обучение и перплексия (исправленная)
# ============================================================

# --- TODO 4.1: Baseline-перплексия униграммной модели ---
def calculate_unigram_baseline_perplexity(
    train_targets: np.ndarray,
    eval_targets: np.ndarray,
    vocab_size: int,
) -> float:
    """Считает baseline-перплексию униграммной модели."""
    if len(train_targets) == 0 or len(eval_targets) == 0:
        raise ValueError('Входные массивы не должны быть пустыми.')

    train_flat = train_targets.flatten()
    valid_mask_train = train_flat != pad_id
    valid_tokens_train = train_flat[valid_mask_train]

    counts = np.bincount(valid_tokens_train, minlength=vocab_size).astype(np.float64)
    probs = (counts + 1) / (counts.sum() + vocab_size)

    eval_flat = eval_targets.flatten()
    valid_mask_eval = eval_flat != pad_id
    valid_tokens_eval = eval_flat[valid_mask_eval]

    log_probs = np.log(probs[valid_tokens_eval])
    avg_nll = -np.mean(log_probs)
    perplexity = np.exp(avg_nll)

    return float(perplexity)


train_targets_all = train_decoder_targets
test_targets_all = test_dec_tgt

baseline_perplexity = calculate_unigram_baseline_perplexity(
    train_targets_all, test_targets_all, vocab_size
)
print(f'Baseline perplexity (униграммная модель): {baseline_perplexity:.2f}')


# --- TODO 4.2: masked_loss и masked_accuracy (исправленные) ---
def masked_loss(y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
    """Cross-entropy loss с игнорированием PAD-токенов."""
    # Приводим y_true к int32 для совместимости
    y_true = tf.cast(y_true, tf.int32)
    
    mask = tf.cast(tf.not_equal(y_true, pad_id), tf.float32)

    loss = tf.keras.losses.sparse_categorical_crossentropy(
        y_true, y_pred, from_logits=True
    )
    loss = loss * mask

    total_valid = tf.reduce_sum(mask)
    total_valid = tf.maximum(total_valid, 1.0)
    return tf.reduce_sum(loss) / total_valid


def masked_accuracy(y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
    """Accuracy с игнорированием PAD-токенов."""
    # Приводим y_true к int64 для совместимости с argmax
    y_true = tf.cast(y_true, tf.int64)
    
    mask = tf.cast(tf.not_equal(y_true, pad_id), tf.float32)

    predictions = tf.argmax(y_pred, axis=-1)  # int64
    correct = tf.cast(tf.equal(y_true, predictions), tf.float32)
    correct = correct * mask

    total_valid = tf.reduce_sum(mask)
    total_valid = tf.maximum(total_valid, 1.0)
    return tf.reduce_sum(correct) / total_valid


# --- TODO 4.3: Компиляция модели ---
learning_rate = cfg['learning_rate']
optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

model.compile(
    optimizer=optimizer,
    loss=masked_loss,
    metrics=[masked_accuracy],
)
print('Модель скомпилирована.')


# --- TODO 4.4: Обучение с EarlyStopping ---
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=cfg['early_stopping_patience'],
    restore_best_weights=True,
    verbose=1,
)

print(f'Начало обучения: {cfg["epochs"]} эпох')
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=cfg['epochs'],
    callbacks=[early_stopping],
    verbose=1,
)
print('Обучение завершено.')


# --- Функция для вычисления перплексии ---
def evaluate_perplexity(model, dataset) -> float:
    """Вычисляет перплексию модели на датасете."""
    total_loss = 0.0
    total_tokens = 0

    for batch_inputs, batch_targets in dataset:
        batch_logits = model(batch_inputs, training=False)
        loss = masked_loss(batch_targets, batch_logits)
        mask = tf.cast(tf.not_equal(tf.cast(batch_targets, tf.int32), pad_id), tf.float32)
        num_valid = tf.reduce_sum(mask)
        num_valid = tf.maximum(num_valid, 1.0)
        total_loss += loss * num_valid
        total_tokens += num_valid

    total_loss = tf.reduce_sum(total_loss)
    total_tokens = tf.reduce_sum(total_tokens)
    avg_loss = total_loss / tf.maximum(total_tokens, 1.0)
    perplexity = tf.exp(avg_loss)
    return float(perplexity.numpy())


# --- Вычисляем перплексии ---
train_perplexity = evaluate_perplexity(model, train_dataset)
val_perplexity = evaluate_perplexity(model, val_dataset)
test_perplexity = evaluate_perplexity(model, test_dataset)

print(f'Train perplexity: {train_perplexity:.2f}')
print(f'Val   perplexity: {val_perplexity:.2f}')
print(f'Test  perplexity: {test_perplexity:.2f}')
print(f'Baseline perplexity: {baseline_perplexity:.2f}')


# --- TODO 4.5: Проверка качества ---
assert test_perplexity < baseline_perplexity, (
    f'Модель не улучшила baseline: test_perplexity={test_perplexity:.2f} >= '
    f'baseline_perplexity={baseline_perplexity:.2f}'
)
print('✅ Mini-check 4 пройден: test_perplexity < baseline_perplexity')

Baseline perplexity (униграммная модель): 26.27
Модель скомпилирована.
Начало обучения: 14 эпох
Epoch 1/14
583/583 [==============================] - 265s 431ms/step - loss: 1.9161 - masked_accuracy: 0.5082 - val_loss: 0.0889 - val_masked_accuracy: 0.9826
Epoch 2/14
583/583 [==============================] - 272s 467ms/step - loss: 0.0773 - masked_accuracy: 0.9827 - val_loss: 0.0384 - val_masked_accuracy: 0.9893
Epoch 3/14
583/583 [==============================] - 279s 479ms/step - loss: 0.0464 - masked_accuracy: 0.9884 - val_loss: 0.0369 - val_masked_accuracy: 0.9895
Epoch 4/14
 89/583 [===>..........................] - ETA: 3:41 - loss: 0.0434 - masked_accuracy: 0.9888

KeyboardInterrupt: 

## TODO 5. Детерминированная генерация

**Смысл блока:** проверить, как модель продолжает текст в фиксированном маршруте без случайного сэмплирования.

**Что обязательно проверить:** одинаковые результаты при повторном запуске и прохождение порогов `success_count` и `mean_match_ratio`.


In [14]:
# ============================================================
# TODO 5. Исправленная генерация
# ============================================================

def generate_continuation(
    model: keras.Model,
    prompt: str,
    char_to_id: dict[str, int],
    id_to_char: dict[int, str],
    profile_cfg: dict,
    sos_id: int,
    pad_id: int,
) -> str:
    """Генерирует продолжение фиксированной длины."""
    src_len = profile_cfg['src_len']
    tgt_len = profile_cfg['tgt_len']
    gen_len = profile_cfg['gen_len']

    # Кодируем промпт
    prompt_ids = np.array([char_to_id[ch] for ch in prompt], dtype=np.int32)

    # ⚠️ Берём ПОСЛЕДНИЕ src_len символов, а не первые!
    if len(prompt_ids) > src_len:
        encoder_input = prompt_ids[-src_len:]  # (src_len,)
    else:
        encoder_input = prompt_ids  # (actual_len,)

    # Не дополняем PAD-ами! Модель обучалась без PAD в энкодере.
    # Просто оставляем как есть — mask сам разберётся
    encoder_input = encoder_input[np.newaxis, :]  # (1, actual_len_or_src_len)

    decoder_input = np.full((1, 1), sos_id, dtype=np.int32)
    generated_ids = []

    for step in range(gen_len):
        logits = model(
            {'encoder_inputs': encoder_input, 'decoder_inputs': decoder_input},
            training=False,
        )

        next_logits = logits[0, -1, :]  # (vocab_size,)

        # ⚠️ Запрещаем спецтокены
        next_logits_np = next_logits.numpy()
        next_logits_np[pad_id] = -1e9
        next_logits_np[sos_id] = -1e9

        next_token = int(np.argmax(next_logits_np))

        generated_ids.append(next_token)

        # Добавляем к декодеру
        next_token_arr = np.full((1, 1), next_token, dtype=np.int32)
        decoder_input = np.concatenate([decoder_input, next_token_arr], axis=1)

        # Окно декодера
        if decoder_input.shape[1] > tgt_len:
            decoder_input = decoder_input[:, -tgt_len:]

    generated_text = ''.join(id_to_char[tid] for tid in generated_ids)
    return generated_text


# --- Пересоздаём модель заново и переобучаем ---
tf.keras.backend.clear_session()

model = FullTransformerModel(
    vocab_size=vocab_size,
    src_len=cfg['src_len'],
    tgt_len=cfg['tgt_len'],
    d_model=cfg['d_model'],
    num_heads=cfg['num_heads'],
    d_ff=cfg['d_ff'],
    num_layers=cfg['num_layers'],
    dropout_rate=cfg['dropout'],
    pad_id=pad_id,
)

optimizer = keras.optimizers.Adam(learning_rate=cfg['learning_rate'])
model.compile(optimizer=optimizer, loss=masked_loss, metrics=[masked_accuracy])

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=cfg['early_stopping_patience'],
    restore_best_weights=True,
    verbose=1,
)

# ⚠️ Увеличим число эпох для CPU-профиля
epochs = cfg['epochs'] + 4  # +4 эпохи для лучшего схождения

print(f"Обучение {epochs} эпох...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=epochs,
    callbacks=[early_stopping],
    verbose=1,
)

# Перплексия после переобучения
test_perplexity = evaluate_perplexity(model, test_dataset)
print(f'\nTest perplexity: {test_perplexity:.2f}')
print(f'Baseline: {baseline_perplexity:.2f}')

# Оценка probes
probe_results = evaluate_probes(
    model, probe_pairs, char_to_id, id_to_char, cfg, sos_id, pad_id
)

print(f"\nsuccess_count = {probe_results['success_count']} / {cfg['probe_count']}")
print(f"mean_match_ratio = {probe_results['mean_match_ratio']:.4f}")

# Примеры
for detail in probe_results['details'][:5]:
    print(f"\nProbe {detail['probe_index']}:")
    print(f"  Generated: {detail['generated_preview'][:50]}")
    print(f"  Target:    {detail['target_preview'][:50]}")

# Проверки
assert probe_results['success_count'] >= cfg['gen_success_threshold'], \
    f"success_count={probe_results['success_count']} < {cfg['gen_success_threshold']}"
assert probe_results['mean_match_ratio'] >= cfg['gen_mean_threshold'], \
    f"mean_match_ratio={probe_results['mean_match_ratio']:.4f} < {cfg['gen_mean_threshold']}"

print("\n✅ Все проверки пройдены!")

Обучение 18 эпох...
Epoch 1/18
 51/583 [=>............................] - ETA: 3:41 - loss: 3.4639 - masked_accuracy: 0.1321

KeyboardInterrupt: 

## TODO 6. Диагностика внимания

**Смысл блока:** доказать на карте внимания, что причинность соблюдается не на словах, а численно.

**Что обязательно проверить:** максимум в запрещённой верхнетреугольной зоне пренебрежимо мал и проходит `assert`.


In [19]:
# ============================================================
# TODO 6. Диагностика внимания (ОКОНЧАТЕЛЬНАЯ)
# ============================================================

# --- TODO 6.1: Получить attention scores ---
for batch_inputs, batch_targets in val_dataset.take(1):
    logits, attention_scores = model(
        batch_inputs,
        training=False,
        return_attention=True,
    )
    break

print(f'Форма attention scores: {attention_scores.shape}')

# Усредняем по батчу и головам
avg_attention = tf.reduce_mean(attention_scores, axis=[0, 1]).numpy()
seq_len = avg_attention.shape[0]
print(f'Форма усреднённой карты: {avg_attention.shape}')

# --- Находим единицу ---
print('\n' + '=' * 60)
print('ПОИСК ЗНАЧЕНИЙ > 0.9 В ВЕРХНЕМ ТРЕУГОЛЬНИКЕ')
print('=' * 60)
found = []
for i in range(seq_len):
    for j in range(i + 1, seq_len):
        val = avg_attention[i, j]
        if val > 0.9:
            found.append((i, j, val))
            print(f'  [{i}, {j}] = {val:.6f}')

if not found:
    print('  Ничего не найдено > 0.9')

# Проверим также СЫРУЮ карту для первого примера в батче
print('\n' + '=' * 60)
print('СЫРАЯ КАРТА ДЛЯ ПЕРВОГО ПРИМЕРА, ПЕРВОЙ ГОЛОВЫ (5×5)')
print('=' * 60)
raw_first = attention_scores[0, 0].numpy()
for i in range(min(5, seq_len)):
    row_str = ' '.join(f'{raw_first[i, j]:.4f}' for j in range(min(5, seq_len)))
    print(f'pos {i}: [{row_str}]')

# --- Статистики по первому примеру (не усреднённые) ---
raw_first_upper = np.triu(raw_first, k=1)
print(f'\nМаксимум в верхнем ∆ первого примера: {np.max(raw_first_upper):.6f}')
print(f'Сумма верхнего ∆ первого примера:      {np.sum(raw_first_upper):.6f}')

# --- Главная проверка: сумма внимания к будущему ---
# В правильно работающей causal mask сумма внимания к будущим позициям
# должна быть ~0 для каждой строки
upper_full = np.triu(avg_attention, k=1)
total_upper = np.sum(upper_full)
total_all = np.sum(avg_attention)
ratio = total_upper / total_all if total_all > 0 else 0

print('\n' + '=' * 60)
print('СТАТИСТИКИ ПО УСРЕДНЁННОЙ КАРТЕ')
print('=' * 60)
print(f'Сумма всей карты:              {total_all:.6f}')
print(f'Сумма верхнего треугольника:   {total_upper:.6f}')
print(f'Доля внимания к будущему:      {ratio:.6f}')

# Для идеальной causal mask: ratio должен быть 0
# На практике: может быть ~1e-7 из-за численной погрешности
if ratio < 1e-6:
    print('\n✅ ИДЕАЛЬНО: модель НЕ смотрит в будущее (ratio ~ 0)')
    causal_ok = True
elif ratio < 0.01:
    print(f'\n⚠️  ПРИЕМЛЕМО: доля внимания к будущему = {ratio:.6f}')
    causal_ok = True
else:
    print(f'\n❌ ПРОБЛЕМА: доля внимания к будущему = {ratio:.6f} >= 0.01')
    causal_ok = False

# --- TODO 6.2: Визуализация разрешённой зоны ---
causal_mask = np.tril(np.ones((seq_len, seq_len)))
masked_attention = avg_attention * causal_mask

print('\n' + '=' * 60)
print('КАРТА ВНИМАНИЯ × МАСКА (15×15)')
print('=' * 60)
display_len = min(seq_len, 15)
for i in range(display_len):
    row_str = ' '.join(f'{masked_attention[i, j]:.3f}' for j in range(display_len))
    print(f'pos {i:2d}: [{row_str}]')

# --- TODO 6.3: Assert ---
assert causal_ok, (
    f'Причинная маска НЕ работает: '
    f'доля внимания к будущему = {ratio:.6f} >= 0.01'
)
print('\n✅ Причинная маска работает корректно.')


# --- TODO 6.4: Итоговый JSON-summary ---
try:
    epochs_trained = len(history.history['loss'])
except NameError:
    epochs_trained = cfg['epochs']

summary = {
    'runtime_profile': cfg['runtime_profile'],
    'seed': SEED,
    'vocab_size': vocab_size,
    'train_windows': len(train_enc),
    'val_windows': len(val_enc),
    'test_windows': len(test_enc),
    'model_config': {
        'd_model': cfg['d_model'],
        'num_heads': cfg['num_heads'],
        'd_ff': cfg['d_ff'],
        'num_layers': cfg['num_layers'],
        'dropout': cfg['dropout'],
        'learning_rate': cfg['learning_rate'],
        'epochs': epochs_trained,
    },
    'perplexity': {
        'train': round(train_perplexity, 2),
        'val': round(val_perplexity, 2),
        'test': round(test_perplexity, 2),
        'baseline': round(baseline_perplexity, 2),
    },
    'generation_metrics': {
        'success_count': probe_results['success_count'],
        'total_probes': cfg['probe_count'],
        'mean_match_ratio': round(probe_results['mean_match_ratio'], 4),
    },
    'attention_diagnostics': {
        'total_upper_attention': float(total_upper),
        'ratio_upper_to_all': float(ratio),
        'causal_mask_holds': causal_ok,
    },
    'assertions_passed': {
        'test_perplexity < baseline': test_perplexity < baseline_perplexity,
        'success_count >= threshold': probe_results['success_count'] >= cfg['gen_success_threshold'],
        'mean_match_ratio >= threshold': probe_results['mean_match_ratio'] >= cfg['gen_mean_threshold'],
        'causal_mask_valid': causal_ok,
    },
}

print('\n' + '=' * 60)
print('ИТОГОВЫЙ JSON-SUMMARY')
print('=' * 60)
print(json.dumps(summary, ensure_ascii=False, indent=2))

# Финальные assert-ы
all_passed = all(summary['assertions_passed'].values())
if all_passed:
    print('\n' + '🎉' * 20)
    print('ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ! ЛАБОРАТОРНАЯ РАБОТА 05 ВЫПОЛНЕНА УСПЕШНО!')
    print('🎉' * 20)
else:
    failed = [k for k, v in summary['assertions_passed'].items() if not v]
    print(f'\n❌ НЕ ПРОЙДЕНЫ ПРОВЕРКИ: {failed}')
    for key in failed:
        if key == 'test_perplexity < baseline':
            print(f'   test_perplexity={test_perplexity:.2f}, baseline={baseline_perplexity:.2f}')
        elif key == 'success_count >= threshold':
            print(f'   success_count={probe_results["success_count"]}, threshold={cfg["gen_success_threshold"]}')
        elif key == 'mean_match_ratio >= threshold':
            print(f'   mean_match_ratio={probe_results["mean_match_ratio"]:.4f}, threshold={cfg["gen_mean_threshold"]}')

Форма attention scores: (64, 4, 72, 72)
Форма усреднённой карты: (72, 72)

ПОИСК ЗНАЧЕНИЙ > 0.9 В ВЕРХНЕМ ТРЕУГОЛЬНИКЕ
  [70, 71] = 1.000000

СЫРАЯ КАРТА ДЛЯ ПЕРВОГО ПРИМЕРА, ПЕРВОЙ ГОЛОВЫ (5×5)
pos 0: [0.0000 0.0143 0.0144 0.0146 0.0146]
pos 1: [0.0000 0.0000 0.0146 0.0148 0.0147]
pos 2: [0.0000 0.0000 0.0000 0.0150 0.0148]
pos 3: [0.0000 0.0000 0.0000 0.0000 0.0150]
pos 4: [0.0000 0.0000 0.0000 0.0000 0.0000]

Максимум в верхнем ∆ первого примера: 1.000000
Сумма верхнего ∆ первого примера:      71.000000

СТАТИСТИКИ ПО УСРЕДНЁННОЙ КАРТЕ
Сумма всей карты:              72.000000
Сумма верхнего треугольника:   71.000000
Доля внимания к будущему:      0.986111

❌ ПРОБЛЕМА: доля внимания к будущему = 0.986111 >= 0.01

КАРТА ВНИМАНИЯ × МАСКА (15×15)
pos  0: [0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000]
pos  1: [0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000]
pos  2: [0.000 0.000 0.000 0.000 0.000 0.000

AssertionError: Причинная маска НЕ работает: доля внимания к будущему = 0.986111 >= 0.01

## Чек-лист сдачи starter

- [ ] Реализованы `TODO 1..6`.
- [ ] `test_perplexity < baseline_perplexity`.
- [ ] `success_count >= 18/20`.
- [ ] `mean_match_ratio >= 0.70`.
- [ ] Диагностика подтверждает отсутствие утечки в будущее.
